In [15]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import yaml
from tqdm import tqdm_notebook
from collections import defaultdict

from data_utils.gecodb_compound_parser import Compound
from data_utils.load_datasets import load_gecodb
from check_applicability import applicability_checkers

applicability_checkers.__init__(celexn_version=1)

In [2]:
rng = np.random.default_rng(579372518)

In [3]:
constr_path = "constraints/constraints_all_revised.yaml"

with open(constr_path, encoding="utf-8") as f:
    yamls = list(yaml.safe_load_all(f))
    constraints: dict = yamls[-1]["constraints"]
    constr_ids = list(constraints.keys())
    data_specs: dict = yamls[-2]["datasets"]

# start with GeCoDB_v07_GN
gecodb = load_gecodb("resources/developed/gecodb_v07_gn.tsv")
N = len(gecodb)

In [21]:
def calc_metrics(golds, preds):
    acc = accuracy_score(golds, preds)
    f1 = f1_score(golds, preds, average="macro")
    return acc, f1

## Baseline: Random Distribution-Aware Sampling

In [10]:
unique_linkers = gecodb["linker"].unique()
linkers = gecodb["linker"].tolist()
linker_distr = pd.Series({
    l: linkers.count(l) / N
    for l in linkers
})
linker_distr.sort_values(ascending=False)

_         0.573485
_+s_      0.268907
_+n_      0.099906
_+en_     0.016232
_+es_     0.015007
_+er_     0.008555
_+e_      0.006452
_+=er_    0.005738
_+ns_     0.003165
_+=e_     0.001940
_+ens_    0.000408
_+=_      0.000204
dtype: float64

In [22]:
rand_linkers = rng.choice(linker_distr.index, size=N, replace=True, p=linker_distr)
calc_metrics(gecodb["linker"], rand_linkers)

(0.4127327670695851, 0.08421396647185787)